# Step 1 : Tokenize the Data

In [ ]:
# Reading the Verdict dataset
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

In [ ]:
# Our goal is to tokenize this 20,479-character short story into individual words
#and special characters that we can then turn into embeddings for LLM training

In [ ]:

import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text) # /s we spilt based on white spaces

print(result)

In [ ]:
result = re.split(r'([,.]|\s)', text)  # split based on , . and white spaces

In [ ]:
print(result)

In [ ]:
result = [item  for item in result if item.strip()] # removing the white spaces

In [ ]:
print(result)

In [ ]:
# #REMOVING WHITESPACES OR NOT
# #When developing a simple tokenizer, whether we should encode whitespaces as
# separate characters or just remove them depends on our application and its requirements.
# Removing whitespaces reduces the memory and computing requirements. 
# However, keeping whitespaces can be useful if we train models that are sensitive to the exact structure of the text (for example, Python code, which is sensitive to indentation and spacing).
# Here, we remove whitespaces for simplicity and brevity of the tokenized outputs. Later, we will switch to a tokenization scheme that includes whitespaces.

In [ ]:
# The tokenization scheme we devised above works well on the simple sample text.
# Let's modify it a bit further so that it can also handle other types of punctuation, such as
# question marks, quotation marks, and the double-dashes
# we have seen earlier in the first 100 characters of Edith Wharton's short story, along with additional special characters

In [ ]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

In [ ]:
# Now that we got a basic tokenizer working, let's apply it to Edith Wharton's entire short story:

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

In [ ]:
print(len(preprocessed))

## Step 2 : Creating Token IDs

In [ ]:
all_words =sorted(set(preprocessed))
print(all_words)

In [ ]:
vocab_size = len(all_words)
print(vocab_size)

In [ ]:
# After determining that the vocabulary size is 1,130 via the above code, we create the vocabulary and print its first 51 entries for illustration purposes:

In [ ]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

In [ ]:
# Let's implement a complete tokenizer class in Python.

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

In [ ]:
tokenizer.decode(ids)

In [ ]:
# So far, so good. We implemented a tokenizer capable of tokenizing and de-tokenizing
# text based on a snippet from the training set. 
# Let's now apply it to a new text sample that is not contained in the training set:

In [ ]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

In [ ]:
# The problem is that the word "Hello" was not used in the The Verdict short story. 
# Hence, it is not contained in the vocabulary. 
# This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs

### ADDING SPECIAL CONTEXT TOKENS

In [ ]:
# In the previous section, we implemented a simple tokenizer and applied it to a passage
# from the training set. 
# In this section, we will modify this tokenizer to handle unknown words.
# In particular, we will modify the vocabulary and tokenizer we implemented in the
# previous section, SimpleTokenizerV2, to support two new tokens, <|unk|> and
# <|endoftext|>

In [ ]:
# We can modify the tokenizer to use an <|unk|> token if it encounters a word that is not part of the vocabulary. 
# Furthermore, we add a token between unrelated texts. 
# For example, when training GPT-like LLMs on multiple independent documents or books, it is common to insert a token before each document or book
# that follows a previous text source

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
print(len(vocab))

In [ ]:
# let's print the last 5 entries of the updated vocabulary:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

In [ ]:
tokenizer.encode(text)


In [ ]:
tokenizer.decode(tokenizer.encode(text))

In [ ]:
# Based on comparing the de-tokenized text above with the original input text,
# we know that the training dataset, Edith Wharton's short story The Verdict,
# did not contain the words "Hello" and "palace."



In [ ]:
# So far, we have discussed tokenization as an essential step in processing text as input to
# LLMs. Depending on the LLM,some researchers also consider additional special tokens such as the following:

# [BOS] (beginning of sequence): This token marks the start of a text. It
# signifies to the LLM where a piece of content begins.

# [EOS] (end of sequence): This token is positioned at the end of a text,
# and is especially useful when concatenating multiple unrelated texts,
# similar to <|endoftext|>. For instance, when combining two different
# Wikipedia articles or books, the [EOS] token indicates where one article
# ends and the next one begins.

# [PAD] (padding): When training LLMs with batch sizes larger than one,
# the batch might contain texts of varying lengths. To ensure all texts have
# the same length, the shorter texts are extended or "padded" using the
# [PAD] token, up to the length of the longest text in the batch.

